# Universidad Politécnica Salesiana
## Carrera de Ciencias de la Computación
### Minería de Datos
#### Periodo 2026-2026
---
Tutor: Ing. Rodolfo Bojorque, Ph.D.

### Práctica 2
#### Elaborado por:
- Integrante 1: Justin Mateo Lucero Reyes
- Integrante 2: Jhonatan Fabricio Tacuri Reinoso
***

# FASE 1: Comprensión del Negocio
## Problema:
En base al dataset asignado ([Matricula UEP Ecuador](https://www.datosabiertos.gob.ec/dataset/base-de-datos-de-matricula-de-uep-ano-2021)) se analiza la movilidad estudiantil en las instituciones de educación superior del Ecuador, identificando estudiantes **foráneos**, es decir, aquellos cuya provincia de residencia difiere de la provincia donde se ubica la sede de su institución. El análisis permite conocer qué carreras y universidades concentran mayor cantidad de estudiantes que se desplazan desde otras provincias para estudiar.

## Detalle de atributos (tanto filas y columnas)
Describir las variables que se analizan y su motivo de porque son necesarias, se recomienda usar esta tabla y formato:

| Variable               | Descripción                                                                                                          | Justificación de su uso                                                                                              | Criterio de eliminación de filas (si aplica)                                                             |
|------------------------|----------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------|
| AÑO                    | Año de registro de la matrícula. Valores enteros en el rango 2015–2023.                                              | Permite analizar la evolución temporal de los estudiantes foráneos e identificar tendencias por año.                 | No se eliminan filas por esta variable; todos los años del dataset son válidos para el análisis.         |
| NOMBRE_IES             | Nombre de la Institución de Educación Superior (universidad o politécnica) donde está matriculado el estudiante.     | Variable principal para filtrar y comparar universidades según la cantidad de estudiantes foráneos que reciben.      | Se eliminan filas con valores nulos, ya que sin institución no es posible ubicar geográficamente la sede.|
| NOMBRE_CARRERA         | Nombre de la carrera o programa académico en el que está matriculado el estudiante.                                  | Permite agrupar y comparar qué carreras concentran mayor número de estudiantes provenientes de otras provincias.     | Se eliminan filas con valores nulos porque la carrera es el eje central del análisis.                    |
| MODALIDAD              | Modalidad de estudio del programa. Valores: Presencial, Semipresencial, En línea, entre otros.                       | Es relevante para el análisis de foráneos: un estudiante en modalidad en línea no necesariamente se desplaza físicamente. | No se eliminan filas; la modalidad se conserva para posibles filtros o análisis comparativos.        |
| PROVINCIA_SEDE         | Provincia donde se encuentra ubicada la sede de la institución educativa.                                            | Variable clave para la comparación geográfica: al contrastarla con PROVINCIA_RESIDENCIA se determina si el estudiante es foráneo. | Se eliminan filas con valores nulos y con valor 'NO_REGISTRA', ya que no permiten determinar la ubicación de la sede. |
| CANTON_SEDE            | Cantón donde se encuentra ubicada la sede de la institución educativa.                                               | Aporta granularidad geográfica sobre la ubicación de las instituciones, útil para análisis a nivel cantonal.         | No se aplica criterio de eliminación directo por esta variable.                                          |
| PROVINCIA_RESIDENCIA   | Provincia de residencia declarada por el estudiante al momento de la matrícula.                                      | Variable clave del análisis: al compararla con PROVINCIA_SEDE se identifica si el estudiante proviene de otra provincia (foráneo). | Se eliminan filas con valores nulos y con valor 'NO_REGISTRA', ya que sin esta información no se puede clasificar al estudiante. |
| CANTON_RESIDENCIA      | Cantón de residencia declarado por el estudiante al momento de la matrícula.                                         | Permite un análisis detallado del origen geográfico de los estudiantes foráneos a nivel cantonal.                    | No se aplica criterio de eliminación directo por esta variable.                                          |
| SEXO                   | Género del estudiante. Valores: Hombre, Mujer.                                                                       | Permite analizar la distribución de género entre los estudiantes foráneos y detectar si existe algún patrón por carrera o institución. | No se eliminan filas; los valores presentes (Hombre/Mujer) son suficientes para el análisis.         |
| TOTAL                  | Cantidad de estudiantes matriculados representados por el registro. Valor numérico entero mayor a cero.              | Es la medida cuantitativa del análisis: permite sumar, agregar y comparar el volumen de estudiantes foráneos.        | No se eliminan filas; registros con TOTAL = 0 no están presentes en el dataset original.                 |

## Fase 2: Limpieza y transformación del dataset

### Carga del dataset

In [1]:
import pandas as pd

# para la carga el dataset deberá estar siempre en la raíz (misma carpeta del libro de Jupyter)
# y no deberá cambiarse el nombre del archivo descargado Base_estadistica_matricula_UEP_15_23.xlsx
file_path = 'Base_estadistica_matricula_UEP_15_23.xlsx'
# La fila 14 contiene los encabezados reales
df_original = pd.read_excel(file_path, sheet_name='Base', header=14)

### Pre-procesamiento
Aquí se podrá usar las celdas que se necesiten para sus transformaciones

In [2]:
# Limpieza de datos: eliminar valores nulos y 'NO_REGISTRA' en variables clave
df_clean = df_original.dropna(subset=['PROVINCIA_SEDE', 'PROVINCIA_RESIDENCIA', 'NOMBRE_CARRERA'])
df_clean = df_clean[(df_clean['PROVINCIA_RESIDENCIA'] != 'NO_REGISTRA') & (df_clean['PROVINCIA_SEDE'] != 'NO_REGISTRA')]

# Filtro de foráneos: estudiantes cuya provincia de residencia no es la misma que la de la sede
df_foraneos = df_clean[df_clean['PROVINCIA_SEDE'] != df_clean['PROVINCIA_RESIDENCIA']].copy()

# Selección de 10 columnas relevantes para análisis de foráneos por carrera
columnas_seleccionadas = [
    'AÑO', 
    'NOMBRE_IES', 
    'NOMBRE_CARRERA', 
    'MODALIDAD', 
    'PROVINCIA_SEDE', 
    'CANTON_SEDE', 
    'PROVINCIA_RESIDENCIA', 
    'CANTON_RESIDENCIA', 
    'SEXO',
    'TOTAL'
]
df_final = df_foraneos[columnas_seleccionadas]

In [3]:
#Use la misma carpeta raíz en el caso de generar archivos auxiliares y para el resultado final
df_final.to_csv('estudiantes_foraneos.csv', index=False)
print(f"Dataset procesado guardado como 'estudiantes_foraneos.csv'")
print(f"Total de registros de foráneos: {len(df_final)}")

Dataset procesado guardado como 'estudiantes_foraneos.csv'
Total de registros de foráneos: 491157


### Previsualización final del dataset
Se deberá mostrar el dataset depurados una vista de las primeras 10 filas es suficiente

In [4]:
# Previsualización de las primeras 10 filas del dataset depurado
df_final.head(10)

,AÑO,NOMBRE_IES,NOMBRE_CARRERA,MODALIDAD,PROVINCIA_SEDE,CANTON_SEDE,PROVINCIA_RESIDENCIA,CANTON_RESIDENCIA,SEXO,TOTAL
60,2015,ESCUELA POLITECNICA NACIONAL,INGENIERIA EN ELECTRONICA Y REDES DE INFORMACION,PRESENCIAL,PICHINCHA,QUITO,ESMERALDAS,ESMERALDAS,MUJER,1
303,2015,ESCUELA POLITECNICA NACIONAL,INGENIERIA CIVIL,PRESENCIAL,PICHINCHA,QUITO,CHIMBORAZO,RIOBAMBA,HOMBRE,1
311,2015,ESCUELA POLITECNICA NACIONAL,INGENIERIA CIVIL,PRESENCIAL,PICHINCHA,QUITO,ESMERALDAS,ESMERALDAS,MUJER,1
812,2015,ESCUELA SUPERIOR POLITECNICA DE CHIMBORAZO,INGENIERIA EN COMERCIO EXTERIOR,PRESENCIAL,CHIMBORAZO,RIOBAMBA,TUNGURAHUA,AMBATO,MUJER,1
817,2015,ESCUELA SUPERIOR POLITECNICA DE CHIMBORAZO,INGENIERIA EN MARKETING,PRESENCIAL,CHIMBORAZO,RIOBAMBA,BOLIVAR,GUARANDA,HOMBRE,1
822,2015,ESCUELA SUPERIOR POLITECNICA DE CHIMBORAZO,INGENIERIA EN MARKETING,PRESENCIAL,CHIMBORAZO,RIOBAMBA,COTOPAXI,PUJILI,HOMBRE,1
823,2015,ESCUELA SUPERIOR POLITECNICA DE CHIMBORAZO,INGENIERIA EN MARKETING,PRESENCIAL,CHIMBORAZO,RIOBAMBA,SANTO DOMINGO DE LOS TSACHILAS,SANTO DOMINGO,HOMBRE,1
824,2015,ESCUELA SUPERIOR POLITECNICA DE CHIMBORAZO,INGENIERIA EN MARKETING,PRESENCIAL,CHIMBORAZO,RIOBAMBA,TUNGURAHUA,AMBATO,HOMBRE,1
826,2015,ESCUELA SUPERIOR POLITECNICA DE CHIMBORAZO,INGENIERIA EN MARKETING,PRESENCIAL,CHIMBORAZO,RIOBAMBA,BOLIVAR,GUARANDA,HOMBRE,2
827,2015,ESCUELA SUPERIOR POLITECNICA DE CHIMBORAZO,INGENIERIA EN MARKETING,PRESENCIAL,CHIMBORAZO,RIOBAMBA,BOLIVAR,SAN MIGUEL,HOMBRE,2
